In [1]:
%load_ext autoreload
%autoreload 2 
import numpy as np 
import shap
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from Preprocess import preprocess_data
from Preprocess import preprocess_data_window
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping
from sklearn.metrics import mean_absolute_error
import optuna
from sklearn.preprocessing import StandardScaler

c:\Users\kaitl\OneDrive\Documents\Icequake Modeling\Code\Icequake-QRC-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:



# Data + preproc 
data_orig = pd.read_csv("../Whillians-GPS-Data-and-Features.csv")
filtered_time = pd.read_csv("../filtered_time_to_next_event.csv")


X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = preprocess_data_window(
    filtered_time,
    data_orig,
    n_previous_events=20
)

# make sure everything is a numpy array 
X_train = X_train.to_numpy()
X_val   = X_val.to_numpy()
X_test  = X_test.to_numpy()

y_train = y_train.to_numpy()
y_val   = y_val.to_numpy()
y_test  = y_test.to_numpy()
# Make sure its scaled 
scaler = StandardScaler()

X_train_np = scaler.fit_transform(np.asarray(X_train))
X_val_np   = scaler.transform(np.asarray(X_val))
X_test_np  = scaler.transform(np.asarray(X_test))

y_train_np = np.asarray(y_train)
y_val_np   = np.asarray(y_val)
y_test_np  = np.asarray(y_test)

# Classic reservoir 
class QuantumComparableReservoir:
    def __init__(
        self,
        input_dim,
        n_features=256,     # ↑ increased capacity
        depth=1,            # ↓ reduce chaos
        noise_level=0.001,  # ↓ stabilize signal
        random_state=42
    ):
        rng = np.random.RandomState(random_state)

        self.input_dim = input_dim
        self.n_features = n_features
        self.depth = depth
        self.noise_level = noise_level
        self.rng = rng

        # Input projection
        self.W_in = rng.normal(0, 1, (n_features, input_dim))
        self.b_in = rng.uniform(0, 2*np.pi, n_features)

        # Reservoir evolution (orthogonal)
        W = rng.randn(n_features, n_features)
        Q, _ = np.linalg.qr(W)
        self.W_res = Q
        self.b_res = rng.uniform(0, 2*np.pi, n_features)

    def transform(self, X):
        X = np.asarray(X)
        outputs = []

        for x in X:
            # Input encoding 
            h = np.sin(self.W_in @ x + self.b_in)

            # Reservoir evolution with residual connections 
            for _ in range(self.depth):
                h_new = np.sin(self.W_res @ h + self.b_res)
                h = 0.7 * h + 0.3 * h_new   

            # Noise (small)
            h += self.noise_level * self.rng.randn(self.n_features)

            # Probability-like mapping
            h = np.exp(h)
            h = h / (np.sum(h) + 1e-8)

            # Restore scale (important for trees)
            h = h * self.n_features

            outputs.append(h)

        outputs = np.array(outputs)

        # Extra nonlinear expansion
        return np.hstack([outputs, outputs**2])


# Apply reservoir 
reservoir = QuantumComparableReservoir(
    input_dim=X_train_np.shape[1],
    n_features=256,
    depth=1,
    noise_level=0.001
)

X_train_res = reservoir.transform(X_train_np)
X_val_res   = reservoir.transform(X_val_np)
X_test_res  = reservoir.transform(X_test_np)

# Augmenting features instead of complete replacement 
X_train_final = np.hstack([X_train_np, X_train_res])
X_val_final   = np.hstack([X_val_np, X_val_res])
X_test_final  = np.hstack([X_test_np, X_test_res])

# Same XGBoost as before 
def objective(trial):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 6),
        subsample=trial.suggest_float("subsample", 0.6, 0.9),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 0.9),
        random_state=42,
        tree_method="hist"
    )

    model.fit(
        X_train_final, y_train_np,
        eval_set=[(X_val_final, y_val_np)],
        verbose=False
    )

    preds = model.predict(X_val_final)
    return root_mean_squared_error(y_val_np, preds)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=30)

optimal_params = study.best_params
print("Optimal Parameters:", optimal_params)


# Final XGBoost 
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    random_state=42,
    tree_method="hist",
    **optimal_params
)

model.fit(
    X_train_final, y_train_np,
    eval_set=[(X_val_final, y_val_np)],
    verbose=False
)

# Looking at error 
preds = model.predict(X_test_final)

rmse = root_mean_squared_error(y_test_np, preds)
mae = mean_absolute_error(y_test_np, preds)

print("Improved Model RMSE:", rmse)
print("Improved Model MAE:", mae)

[I 2026-03-19 14:14:14,319] A new study created in memory with name: no-name-e4d4f7e9-133d-4166-a321-4dfd543b87ae


X shape:  (2822, 126)
y shape:  (2822,)


[I 2026-03-19 14:14:47,337] Trial 0 finished with value: 17566.875107685053 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 6, 'subsample': 0.8195981825434215, 'colsample_bytree': 0.779597545259111}. Best is trial 0 with value: 17566.875107685053.
[I 2026-03-19 14:14:55,581] Trial 1 finished with value: 17808.06613337592 and parameters: {'learning_rate': 0.014322493718230255, 'max_depth': 3, 'subsample': 0.6174250836504598, 'colsample_bytree': 0.8598528437324806}. Best is trial 0 with value: 17566.875107685053.
[I 2026-03-19 14:15:15,493] Trial 2 finished with value: 17728.581712153482 and parameters: {'learning_rate': 0.039913058785616795, 'max_depth': 5, 'subsample': 0.6061753482887408, 'colsample_bytree': 0.8909729556485984}. Best is trial 0 with value: 17566.875107685053.
[I 2026-03-19 14:15:23,202] Trial 3 finished with value: 18093.73031633426 and parameters: {'learning_rate': 0.06798962421591129, 'max_depth': 3, 'subsample': 0.6545474901621302, 'colsample_by

Optimal Parameters: {'learning_rate': 0.017470961538297445, 'max_depth': 5, 'subsample': 0.7981797737485374, 'colsample_bytree': 0.6334862200260294}
Improved Model RMSE: 16358.697841057687
Improved Model MAE: 13062.701804480088


In [ ]:
#Data and preproc 
data_orig = pd.read_csv("../Whillians-GPS-Data-and-Features.csv")
filtered_time = pd.read_csv("../filtered_time_to_next_event.csv")

X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = preprocess_data_window(
    filtered_time,
    data_orig,
    n_previous_events=20
)

# Make sure everything is numpy
X_train = X_train.to_numpy()
X_val   = X_val.to_numpy()
X_test  = X_test.to_numpy()

y_train = y_train.to_numpy()
y_val   = y_val.to_numpy()
y_test  = y_test.to_numpy()

# Make sure everything is scaled 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

#Creating classic reservoir 
class QuantumComparableReservoir:
    def __init__(
        self,
        input_dim,
        n_features=64,   # match 2^n_qubits
        depth=2,
        noise_level=0.01,
        random_state=42
    ):
        rng = np.random.RandomState(random_state)

        self.input_dim = input_dim
        self.n_features = n_features
        self.depth = depth
        self.noise_level = noise_level
        self.rng = rng

        # Projecting to higher dimensional space 
        self.W_in = rng.normal(0, 1, (n_features, input_dim))
        self.b_in = rng.uniform(0, 2*np.pi, n_features)

        # Reservoir evolving 
        W = rng.randn(n_features, n_features)
        Q, _ = np.linalg.qr(W)  # orthogonal 
        self.W_res = Q
        self.b_res = rng.uniform(0, 2*np.pi, n_features)

    def transform(self, X):
        X = np.asarray(X)
        outputs = []

        for x in X:
            # Input encoding kind of like quantum Ry 
            h = np.sin(self.W_in @ x + self.b_in)

            # Reservoir layers 
            for _ in range(self.depth):
                h = np.sin(self.W_res @ h + self.b_res)

            # Noise amount 
            h += self.noise_level * self.rng.randn(self.n_features)

            # Measurement---compareable to final measurement in quantum 
            h = np.exp(h)
            h = h / (np.sum(h) + 1e-8)

            outputs.append(h)

        return np.array(outputs)

# Applying reservour 
reservoir = QuantumComparableReservoir(input_dim=X_train.shape[1])

X_train_res = reservoir.transform(X_train)
X_val_res   = reservoir.transform(X_val)
X_test_res  = reservoir.transform(X_test)

# Combining the features to be fed into XGBoost
X_train_final = np.hstack([X_train, X_train_res])
X_val_final   = np.hstack([X_val, X_val_res])
X_test_final  = np.hstack([X_test, X_test_res])


# Same XGBoost as before 
def objective(trial):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        subsample=trial.suggest_float("subsample", 0.6, 0.9),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 0.9),
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_final,
        y_train,
        eval_set=[(X_val_final, y_val)],
        verbose=False
    )

    preds = model.predict(X_val_final)
    rmse = root_mean_squared_error(y_val, preds)
    return rmse

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30)

print("Optimal Parameters:", study.best_params)


# Final XGBoost
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_final,
    y_train,
    eval_set=[(X_val_final, y_val)],
    verbose=False
)

# Looking at error 
preds = model.predict(X_test_final)

rmse = root_mean_squared_error(y_test, preds)
mae  = mean_absolute_error(y_test, preds)

print("XGBoost RMSE:", rmse)
print("XGBoost MAE:", mae)

[I 2026-03-19 14:09:14,774] A new study created in memory with name: no-name-097eb85d-1649-48bc-aaba-d318efe7968c


X shape:  (2822, 126)
y shape:  (2822,)


[I 2026-03-19 14:09:21,322] Trial 0 finished with value: 16961.273165505903 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 5, 'subsample': 0.8195981825434215, 'colsample_bytree': 0.779597545259111}. Best is trial 0 with value: 16961.273165505903.
[I 2026-03-19 14:09:22,362] Trial 1 finished with value: 17795.956682967328 and parameters: {'learning_rate': 0.014322493718230255, 'max_depth': 2, 'subsample': 0.6174250836504598, 'colsample_bytree': 0.8598528437324806}. Best is trial 0 with value: 16961.273165505903.
[I 2026-03-19 14:09:25,129] Trial 2 finished with value: 17310.4711996595 and parameters: {'learning_rate': 0.039913058785616795, 'max_depth': 4, 'subsample': 0.6061753482887408, 'colsample_bytree': 0.8909729556485984}. Best is trial 0 with value: 16961.273165505903.
[I 2026-03-19 14:09:26,094] Trial 3 finished with value: 18024.182228218448 and parameters: {'learning_rate': 0.06798962421591129, 'max_depth': 2, 'subsample': 0.6545474901621302, 'colsample_by

Optimal Parameters: {'learning_rate': 0.012542465200210231, 'max_depth': 5, 'subsample': 0.8675598384414251, 'colsample_bytree': 0.7611831496838132}
XGBoost RMSE: 15706.651135983637
XGBoost MAE: 12218.461255530974
